# Notebook B: Combining /accounts, /groups, and /users

**Purpose:** Pull records from three related FOLIO endpoints and join them into a
single analysis-ready DataFrame. This is the kind of cross-record combination the
Stripes UI doesn't do natively.

**Assumed relationships** (confirm against your instance — I have not verified these
field names against current Sunflower schema, so treat them as a starting hypothesis
to check, not a fact):
- `/accounts` records reference a user via a `userId` field.
- `/users` records reference a patron group via a `patronGroup` field (a group's `id`).
- `/groups` records have an `id` and a human-readable `group` name.

If any of those field names are wrong for your instance, the fix is just changing the
column name in the merge step below — the overall approach stays the same.

**How to use this notebook:** Each section has a markdown cell explaining the step,
followed by a code cell. `# TODO (FOLIO-specific)` marks spots to confirm/adjust
against your actual schema.


## 1. Environment setup


In [ ]:
# !pip install pandas requests

import pandas as pd
import requests
from datetime import date
pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [ ]:

%run folio_auth.ipynb


## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [ ]:
def fetch_all_records(endpoint, records_key, limit=100, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records

## 4. Pull data from each endpoint

In [ ]:

accounts_raw = fetch_all_records("/accounts", query='status.name="Open" and remaining > 0', records_key="accounts")
groups_raw   = fetch_all_records("/groups",   records_key="usergroups")  
users_raw    = fetch_all_records("/users",    records_key="users")

print(f"accounts: {len(accounts_raw)}")
print(f"groups:   {len(groups_raw)}")
print(f"users:    {len(users_raw)}")


In [ ]:
accounts_df = pd.DataFrame(accounts_raw)
groups_df   = pd.DataFrame(groups_raw)
users_df    = pd.DataFrame(users_raw)

users_df.head()


## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?

In [ ]:
print(accounts_df.columns.tolist())
print(users_df.columns.tolist())
print(groups_df.columns.tolist())


In [ ]:
# Spot-check types of the columns you intend to join on
print(accounts_df['userId'].dtype)
print(users_df['id'].dtype)
print(users_df['patronGroup'].dtype)
print(groups_df['id'].dtype)


## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [ ]:
# Step 1: accounts + users
accounts_users = accounts_df.merge(
    users_df,
    left_on='userId',
    right_on='id',
    how='left',
    suffixes=('_account', '_user'),
)

# Step 2: + groups
full_df = accounts_users.merge(
    groups_df,
    left_on='patronGroup',
    right_on='id',
    how='left',
    suffixes=('', '_group'),
)

full_df.head()


## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [ ]:
print("Original accounts rows:", len(accounts_df))
print("After merging with users:  ", len(accounts_users))
print("After merging with groups: ", len(full_df))

# Rows where the user or group match failed — worth investigating, not ignoring
unmatched_users = full_df[full_df['id_user'].isnull()] if 'id_user' in full_df.columns else pd.DataFrame()
print("Accounts with no matching user:", len(unmatched_users))


## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


### This call creates a summary of total fines by patron group

In [ ]:
summary = full_df.groupby('group')['remaining'].sum().sort_values(ascending=False)
print(summary)


In [ ]:
# fees_fines_report = full_df[['username', 'remaining', 'feeFineType']]
full_df.head()

In [ ]:

fees_fines_report = full_df.loc[:, ['barcode_user','amount','remaining', 'feeFineType']]
fees_fines_report['lastName']= full_df['personal'].apply(lambda x: x.get('lastName') if isinstance(x,dict) else None)
fees_fines_report['firstName']= full_df['personal'].apply(lambda x: x.get('firstName') if isinstance(x,dict) else None)
fees_fines_report['preferredFirstName']= full_df['personal'].apply(lambda x: x.get('preferredFirstName') if isinstance(x,dict) else None)
fees_fines_report['email']= full_df['personal'].apply(lambda x: x.get('email') if isinstance(x,dict) else None)
fees_fines_report['item-title']=full_df['title']

fees_fines_report.head()

In [ ]:
today = date.today()
formatted_date = today.strftime("%Y-%m-%d")
fees_fines_report.to_csv('Fees & Fines-' + formatted_date + '.csv', index=False)